In [6]:
import pandas as pd

X_train = pd.read_parquet("../data/X_train_final.parquet")
y_train = pd.read_parquet("../data/y_train_final.parquet")

X_test = pd.read_parquet("../data/X_test_final.parquet")
y_test = pd.read_parquet("../data/y_test_final.parquet")

# Model building

I will build multiple models to evaluate their performance against one another so I can pick the best one for this usecase. I will start with a simple baseline model, and will choose Logistic Regression since we have a binary output. I will also start by using all features of the dataset to get a baseline model performance.

In [7]:
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    RocCurveDisplay,
)

log_reg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

log_reg.fit(X_train, y_train)

y_test_eval = y_test.squeeze()

y_pred = log_reg.predict(X_test)
y_proba = log_reg.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test_eval, y_pred))
print("Precision:", precision_score(y_test_eval, y_pred))
print("Recall:", recall_score(y_test_eval, y_pred))
print("F1 Score:", f1_score(y_test_eval, y_pred))
print("ROC AUC Score:", roc_auc_score(y_test_eval, y_pred))


Accuracy: 0.7381121362668559
Precision: 0.504302925989673
Recall: 0.7834224598930482
F1 Score: 0.6136125654450262
ROC AUC Score: 0.7525807951639154


/home/jaydanng/Projects/churn-prediction-system/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:1352: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


From these scores above, we can see that the model has fairly high recall of $0.783$ meaning that the model is pretty good at catching most of the churners. The downside of this is the low precision score of $0.504$ meaning the model also flags a lot of non churners as churners (false positives). Next we want to look into the features a little more to see what we can learn.

In [11]:
coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": log_reg.coef_[0]
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()

coef_df = coef_df.sort_values(
    by="abs_coefficient",
    ascending=False
)

coef_df.head(15)

,feature,coefficient,abs_coefficient
1,MonthlyCharges,-1.123589,1.123589
38,Contract_Two year,-0.781257,0.781257
16,InternetService_Fiber optic,0.705336,0.705336
2,TotalCharges,-0.663581,0.663581
36,Contract_Month-to-month,0.663399,0.663399
15,InternetService_DSL,-0.621009,0.621009
3,SeniorCitizen,0.457211,0.457211
35,StreamingMovies_Yes,0.274978,0.274978
25,DeviceProtection_No internet service,-0.274568,0.274568
17,InternetService_No,-0.274568,0.274568


From this we can see a few key insights:
1. First, we can see that the contract type is a big driver of churn. Customers with long term contracts are much less likely to churn compared to those with short term contracts (Two year vs Month to Month). 
2. Second, we can see that customers with fiber optic internet are much more likely to churn compared to DSL users.
3. Third, we can see the effect of tenure via the TotalCharges which shows once again that long term customers are much less likely to churn.
4. Finally, we see that senior citizens show a higher likelihood of churn

In this data, while monthly charges seems to be negatively associated with churn in the model, this is most likely due to interactions with other variables such as contract type and internet service. This just shows the importance of considering issues with multicollinearity when dealing with linear models.

The model identified contract type, internet service type, and customer tenure as the strongest predictors of churn. Customers on month-to-month contracts and those using fiber optic internet were significantly more likely to churn, while long-term contract holders and high-tenure customers showed strong retention. Additionally, senior customers exhibited higher churn risk. Some coefficients, such as monthly charges, reflect underlying correlations with other features, highlighting the impact of multicollinearity in linear models.